# Heart Disease — Classification

**Dataset:** UCI Heart Disease (918 records)  
**Target:** `HeartDisease` (binary)  
**Models:** Logistic Regression, KNN, Naive Bayes, Decision Tree, SVM  
**Flow:** Preprocess → Train/Test Split → Train 5 models → Compare Accuracy & F1 → Save best model

## 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 2. Load & Preprocess

In [ ]:
df = pd.read_csv('../datasets/heart.csv')

# Fix biologically impossible zeros
for col in ['Cholesterol', 'RestingBP']:
    col_mean = df.loc[df[col] != 0, col].mean()
    df[col] = df[col].replace(0, col_mean).round(2)

# One-hot encode
df_encoded = pd.get_dummies(df, drop_first=True).astype(int)
df_encoded.head()

## 3. Feature / Target Split & Train-Test Split

In [ ]:
X = df_encoded.drop('HeartDisease', axis=1)
y = df_encoded['HeartDisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 4. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # transform only — no fit on test

## 5. Train & Evaluate All Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'KNN':                 KNeighborsClassifier(),
    'Naive Bayes':         GaussianNB(),
    'Decision Tree':       DecisionTreeClassifier(random_state=42),
    'SVM':                 SVC(kernel='rbf')
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred   = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    f1       = f1_score(y_test, y_pred)
    results.append({'Model': name, 'Accuracy': round(accuracy, 4), 'F1 Score': round(f1, 4)})

results_df = pd.DataFrame(results).sort_values('F1 Score', ascending=False)
results_df

## 6. Best Model — Detailed Report

In [ ]:
best_name = results_df.iloc[0]['Model']
best_model = models[best_name]
y_pred_best = best_model.predict(X_test_scaled)

print(f'Best Model: {best_name}\n')
print(classification_report(y_test, y_pred_best, target_names=['No Disease', 'Heart Disease']))

## 7. Save Model & Artifacts

In [ ]:
model_dir = 'models'

# Create the folder automatically if it does not exist
os.makedirs(model_dir, exist_ok=True)

# Save your model files into the folder
joblib.dump(best_model, os.path.join(model_dir, 'heart_best_model.pkl'))
joblib.dump(scaler,     os.path.join(model_dir, 'heart_scaler.pkl'))
joblib.dump(X.columns.tolist(), os.path.join(model_dir, 'heart_feature_columns.pkl'))

print(f'Saved: {best_name}')
